In [1]:
"""
=============================================================================
 TRANSFORMER FROM SCRATCH — Complete Implementation in PyTorch
=============================================================================
 Every component (Scaled Dot-Product Attention, Multi-Head Attention,
 Positional Encoding, Feed-Forward Network, Encoder, Decoder) is built
 from first principles. Trained on a small synthetic EN→FR dataset so
 the script is fully self-contained — no external downloads needed.
"""

import math
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ── reproducibility ─────────────────────────────────────────────────────────
torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ============================================================================
# 1.  SCALED DOT-PRODUCT ATTENTION
# ============================================================================
#   Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
#
#   • Q, K, V  — query, key, value matrices
#   • d_k      — dimension of keys (used to scale the dot products)
#   • mask     — optional mask to prevent attending to certain positions
# ============================================================================

def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
    """
    Args
    ----
    query : (batch, heads, seq_len_q, d_k)
    key   : (batch, heads, seq_len_k, d_k)
    value : (batch, heads, seq_len_v, d_v)   [seq_len_k == seq_len_v]
    mask  : broadcastable to (batch, 1, seq_len_q, seq_len_k)

    Returns
    -------
    output          : (batch, heads, seq_len_q, d_v)
    attention_weights: (batch, heads, seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)

    # Step 1 — dot product between Q and K^T
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2 — apply mask (for padding or causal / look-ahead masking)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-1e9"))

    # Step 3 — softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention_weights = dropout(attention_weights)

    # Step 4 — weighted sum of values
    output = torch.matmul(attention_weights, value)
    return output, attention_weights


# ============================================================================
# 2.  MULTI-HEAD ATTENTION
# ============================================================================
#   Instead of one attention function, we project Q, K, V  h  times with
#   different learned linear projections, run attention in parallel, then
#   concatenate and project again.
# ============================================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads   # dimension per head

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1) Linear projections  →  (batch, heads, seq_len, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 2) Apply attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask, self.dropout)

        # 3) Concatenate heads  →  (batch, seq_len, d_model)
        attn_output = (
            attn_output.transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.d_model)
        )

        # 4) Final linear projection
        return self.W_o(attn_output)


# ============================================================================
# 3.  POSITION-WISE FEED-FORWARD NETWORK
# ============================================================================
#   FFN(x) = ReLU(x·W₁ + b₁)·W₂ + b₂
#   Applied independently to each position.
# ============================================================================

class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


# ============================================================================
# 4.  POSITIONAL ENCODING  (sinusoidal — no learnable parameters)
# ============================================================================
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
# ============================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)                       # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float() # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices
        pe[:, 1::2] = torch.cos(position * div_term)   # odd  indices

        pe = pe.unsqueeze(0)  # (1, max_len, d_model) for broadcasting
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# ============================================================================
# 5.  ENCODER LAYER  &  ENCODER STACK
# ============================================================================

class EncoderLayer(nn.Module):
    """One layer: Self-Attention → Add & Norm → FFN → Add & Norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn        = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.dropout1   = nn.Dropout(dropout)
        self.dropout2   = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        # Sub-layer 1: multi-head self-attention
        attn_out = self.self_attn(src, src, src, src_mask)
        src = self.norm1(src + self.dropout1(attn_out))

        # Sub-layer 2: feed-forward
        ffn_out = self.ffn(src)
        src = self.norm2(src + self.dropout2(ffn_out))
        return src


class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.d_model = d_model

    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return x


# ============================================================================
# 6.  DECODER LAYER  &  DECODER STACK
# ============================================================================

class DecoderLayer(nn.Module):
    """Masked Self-Attn → Add&Norm → Cross-Attn → Add&Norm → FFN → Add&Norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn   = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn         = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        # Sub-layer 1: masked self-attention (causal)
        attn_out = self.self_attn(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(tgt + self.dropout1(attn_out))

        # Sub-layer 2: cross-attention over encoder output
        attn_out = self.cross_attn(tgt, enc_output, enc_output, src_mask)
        tgt = self.norm2(tgt + self.dropout2(attn_out))

        # Sub-layer 3: feed-forward
        ffn_out = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout3(ffn_out))
        return tgt


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.layers    = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.d_model = d_model

    def forward(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x


# ============================================================================
# 7.  FULL TRANSFORMER  (Encoder + Decoder + Output Projection)
# ============================================================================

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        num_heads=8,
        d_ff=512,
        num_layers=3,
        dropout=0.1,
        max_len=100,
    ):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads, d_ff, num_layers, dropout)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        logits     = self.output_projection(dec_output)
        return logits

    # ── mask utilities ──────────────────────────────────────────────────────
    @staticmethod
    def make_src_mask(src, pad_idx):
        """(batch, 1, 1, src_len) — masks <PAD> tokens"""
        return (src != pad_idx).unsqueeze(1).unsqueeze(2)

    @staticmethod
    def make_tgt_mask(tgt, pad_idx):
        """Combines padding mask with causal (look-ahead) mask."""
        batch, tgt_len = tgt.shape
        pad_mask = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)          # (B,1,1,T)
        causal   = torch.tril(torch.ones(tgt_len, tgt_len, device=tgt.device)).bool()  # (T,T)
        return pad_mask & causal  # (B,1,T,T) via broadcast


# ============================================================================
# 8.  VOCABULARY & TOKENIZER  (simple word-level)
# ============================================================================

class SimpleVocab:
    PAD_TOKEN = "<PAD>"
    SOS_TOKEN = "<SOS>"
    EOS_TOKEN = "<EOS>"
    UNK_TOKEN = "<UNK>"
    SPECIAL   = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

    def __init__(self, sentences, min_freq=1):
        counter = Counter()
        for s in sentences:
            counter.update(s.lower().split())
        self.itos = list(self.SPECIAL) + [w for w, c in counter.items() if c >= min_freq]
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, sentence):
        tokens = [self.stoi.get(w, self.stoi[self.UNK_TOKEN]) for w in sentence.lower().split()]
        return [self.stoi[self.SOS_TOKEN]] + tokens + [self.stoi[self.EOS_TOKEN]]

    def decode(self, indices):
        return " ".join(
            self.itos[i] for i in indices
            if self.itos[i] not in {self.PAD_TOKEN, self.SOS_TOKEN, self.EOS_TOKEN}
        )

    @property
    def pad_idx(self):
        return self.stoi[self.PAD_TOKEN]

    def __len__(self):
        return len(self.itos)


# ============================================================================
# 9.  SYNTHETIC EN→FR DATASET  (self-contained, no downloads)
# ============================================================================

PARALLEL_DATA = [
    ("the cat sits on the mat",           "le chat est assis sur le tapis"),
    ("i love this movie",                 "j'aime ce film"),
    ("she is reading a book",             "elle lit un livre"),
    ("they are playing in the park",      "ils jouent dans le parc"),
    ("he runs every morning",             "il court chaque matin"),
    ("we eat breakfast together",         "nous prenons le petit déjeuner ensemble"),
    ("the dog is sleeping",              "le chien dort"),
    ("she writes beautiful poems",        "elle écrit de beaux poèmes"),
    ("the sun is shining brightly",       "le soleil brille intensément"),
    ("i am learning french",              "j'apprends le français"),
    ("he plays the guitar well",          "il joue bien de la guitare"),
    ("the children laugh happily",        "les enfants rient joyeusement"),
    ("we travel to paris",                "nous voyageons à paris"),
    ("they study mathematics",            "ils étudient les mathématiques"),
    ("the flowers bloom in spring",       "les fleurs fleurissent au printemps"),
    ("she drinks coffee every day",       "elle boit du café chaque jour"),
    ("i watch the stars at night",        "je regarde les étoiles la nuit"),
    ("the bird sings a song",             "l'oiseau chante une chanson"),
    ("he reads the newspaper",            "il lit le journal"),
    ("we walk along the river",           "nous marchons le long de la rivière"),
    ("the teacher explains the lesson",   "le professeur explique la leçon"),
    ("she paints a beautiful picture",    "elle peint un beau tableau"),
    ("they cook dinner together",         "ils préparent le dîner ensemble"),
    ("the rain falls softly",             "la pluie tombe doucement"),
    ("i write a letter to my friend",     "j'écris une lettre à mon ami"),
    ("he swims in the ocean",             "il nage dans l'océan"),
    ("we listen to music",                "nous écoutons de la musique"),
    ("the baby sleeps peacefully",        "le bébé dort paisiblement"),
    ("she dances with grace",             "elle danse avec grâce"),
    ("they build a sandcastle",           "ils construisent un château de sable"),
]


class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab, max_len=20):
        self.pairs     = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len   = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_sent, tgt_sent = self.pairs[idx]
        src_ids = self.src_vocab.encode(src_sent)[: self.max_len]
        tgt_ids = self.tgt_vocab.encode(tgt_sent)[: self.max_len]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    return src_padded, tgt_padded


# ============================================================================
# 10.  TRAINING LOOP
# ============================================================================

def train_epoch(model, dataloader, optimizer, criterion, src_vocab, tgt_vocab):
    model.train()
    total_loss = 0
    for src, tgt in dataloader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        tgt_input  = tgt[:, :-1]   # teacher forcing: input  = <SOS> ... t_{n-1}
        tgt_output = tgt[:, 1:]    # expected output           = t_1  ... <EOS>

        src_mask = Transformer.make_src_mask(src, src_vocab.pad_idx).to(DEVICE)
        tgt_mask = Transformer.make_tgt_mask(tgt_input, tgt_vocab.pad_idx).to(DEVICE)

        logits = model(src, tgt_input, src_mask, tgt_mask)  # (B, T, V)
        logits = logits.reshape(-1, logits.size(-1))
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(logits, tgt_output)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)


# ============================================================================
# 11.  GREEDY DECODE  (inference)
# ============================================================================

@torch.no_grad()
def greedy_decode(model, src_sentence, src_vocab, tgt_vocab, max_len=20):
    model.eval()
    src_ids  = torch.tensor([src_vocab.encode(src_sentence)]).to(DEVICE)
    src_mask = Transformer.make_src_mask(src_ids, src_vocab.pad_idx).to(DEVICE)
    enc_out  = model.encoder(src_ids, src_mask)

    tgt_ids = [tgt_vocab.stoi[SimpleVocab.SOS_TOKEN]]

    for _ in range(max_len):
        tgt_tensor = torch.tensor([tgt_ids]).to(DEVICE)
        tgt_mask   = Transformer.make_tgt_mask(tgt_tensor, tgt_vocab.pad_idx).to(DEVICE)
        dec_out    = model.decoder(tgt_tensor, enc_out, src_mask, tgt_mask)
        logits     = model.output_projection(dec_out[:, -1, :])
        next_token = logits.argmax(dim=-1).item()
        tgt_ids.append(next_token)
        if next_token == tgt_vocab.stoi[SimpleVocab.EOS_TOKEN]:
            break

    return tgt_vocab.decode(tgt_ids)


# ============================================================================
# 12.  MAIN — BUILD, TRAIN, EVALUATE
# ============================================================================

def main():
    print("=" * 70)
    print("  TRANSFORMER FROM SCRATCH — EN → FR Translation Demo")
    print("=" * 70)

    # ── build vocabs ────────────────────────────────────────────────────────
    src_sents = [p[0] for p in PARALLEL_DATA]
    tgt_sents = [p[1] for p in PARALLEL_DATA]
    src_vocab = SimpleVocab(src_sents)
    tgt_vocab = SimpleVocab(tgt_sents)
    print(f"\nSource vocab size : {len(src_vocab)}")
    print(f"Target vocab size : {len(tgt_vocab)}")

    # ── dataset & dataloader ────────────────────────────────────────────────
    dataset    = TranslationDataset(PARALLEL_DATA, src_vocab, tgt_vocab)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

    # ── model ───────────────────────────────────────────────────────────────
    D_MODEL    = 128
    NUM_HEADS  = 4
    D_FF       = 256
    NUM_LAYERS = 2
    DROPOUT    = 0.1

    model = Transformer(
        src_vocab_size=len(src_vocab),
        tgt_vocab_size=len(tgt_vocab),
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        d_ff=D_FF,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters  : {total_params:,}")

    # ── optimizer & loss ────────────────────────────────────────────────────
    optimizer = optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

    # ── training ────────────────────────────────────────────────────────────
    NUM_EPOCHS = 80
    print(f"\nTraining for {NUM_EPOCHS} epochs ...\n")
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        loss = train_epoch(model, dataloader, optimizer, criterion, src_vocab, tgt_vocab)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{NUM_EPOCHS}  |  Loss: {loss:.4f}")

    elapsed = time.time() - t0
    print(f"\nTraining completed in {elapsed:.1f}s")

    # ── inference demo ──────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  TRANSLATION RESULTS  (greedy decoding)")
    print("=" * 70 + "\n")

    test_sentences = [
        "the cat sits on the mat",
        "i love this movie",
        "she is reading a book",
        "the sun is shining brightly",
        "we walk along the river",
        "the bird sings a song",
    ]

    for sent in test_sentences:
        translation = greedy_decode(model, sent, src_vocab, tgt_vocab)
        print(f"  EN: {sent}")
        print(f"  FR: {translation}\n")

    # ── architecture summary ────────────────────────────────────────────────
    print("=" * 70)
    print("  ARCHITECTURE SUMMARY")
    print("=" * 70)
    print(f"""
    ┌─────────────────────────────────────────────────────┐
    │                  TRANSFORMER                        │
    │                                                     │
    │  ┌──────────────────┐   ┌──────────────────┐       │
    │  │     ENCODER      │   │     DECODER      │       │
    │  │  (x{NUM_LAYERS} layers)     │   │  (x{NUM_LAYERS} layers)     │       │
    │  │                  │   │                  │       │
    │  │ ┌──────────────┐ │   │ ┌──────────────┐ │       │
    │  │ │  Self-Attn   │ │   │ │ Masked Self  │ │       │
    │  │ │  ({NUM_HEADS} heads)   │ │   │ │  Attention   │ │       │
    │  │ │  d_k={D_MODEL // NUM_HEADS}       │ │   │ │  ({NUM_HEADS} heads)   │ │       │
    │  │ └──────────────┘ │   │ └──────────────┘ │       │
    │  │ ┌──────────────┐ │   │ ┌──────────────┐ │       │
    │  │ │ Add & Norm   │ │   │ │ Cross-Attn   │ │       │
    │  │ └──────────────┘ │   │ │  ({NUM_HEADS} heads)   │ │       │
    │  │ ┌──────────────┐ │   │ └──────────────┘ │       │
    │  │ │   FFN        │ │   │ ┌──────────────┐ │       │
    │  │ │ {D_MODEL}→{D_FF}→{D_MODEL} │ │   │ │   FFN        │ │       │
    │  │ └──────────────┘ │   │ │ {D_MODEL}→{D_FF}→{D_MODEL} │ │       │
    │  │ ┌──────────────┐ │   │ └──────────────┘ │       │
    │  │ │ Add & Norm   │ │   │ ┌──────────────┐ │       │
    │  │ └──────────────┘ │   │ │ Linear→Vocab │ │       │
    │  └──────────────────┘   │ └──────────────┘ │       │
    │                         └──────────────────┘       │
    │                                                     │
    │  Positional Encoding: Sinusoidal (no learnable)    │
    │  Total Parameters  : {total_params:,}                  │
    └─────────────────────────────────────────────────────┘
    """)


if __name__ == "__main__":
    main()

Using device: cpu
  TRANSFORMER FROM SCRATCH — EN → FR Translation Demo

Source vocab size : 97
Target vocab size : 101
Model parameters  : 700,901

Training for 80 epochs ...

  Epoch   1/80  |  Loss: 4.6869
  Epoch  10/80  |  Loss: 3.0175
  Epoch  20/80  |  Loss: 1.8353
  Epoch  30/80  |  Loss: 0.9912
  Epoch  40/80  |  Loss: 0.5492
  Epoch  50/80  |  Loss: 0.2675
  Epoch  60/80  |  Loss: 0.1490
  Epoch  70/80  |  Loss: 0.0822
  Epoch  80/80  |  Loss: 0.0543

Training completed in 11.8s

  TRANSLATION RESULTS  (greedy decoding)

  EN: the cat sits on the mat
  FR: le chat est assis sur le tapis

  EN: i love this movie
  FR: j'aime ce film

  EN: she is reading a book
  FR: elle lit un livre

  EN: the sun is shining brightly
  FR: le soleil brille intensément

  EN: we walk along the river
  FR: nous marchons le long de la rivière

  EN: the bird sings a song
  FR: l'oiseau chante une chanson

  ARCHITECTURE SUMMARY

    ┌─────────────────────────────────────────────────────┐
    │ 